In [1]:
import numpy as np
from tensorflow.keras.datasets import mnist

In [2]:
# 1. LOAD AND PREP DATA (Same as before)
(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train = x_train.reshape(x_train.shape[0], 784) / 255.0
x_test = x_test.reshape(x_test.shape[0], 784) / 255.0

def one_hot(y):
    one_hoty = np.zeros((y.size, 10))
    one_hoty[np.arange(y.size), y] = 1
    return one_hoty

y_train_encod = one_hot(y_train)
y_test_encod = one_hot(y_test)

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


In [3]:
# 2. ACTIVATION FUNCTIONS
def relu(Z):
    return np.maximum(0, Z)

def relu_derivative(Z):
    return Z > 0

def softmax(Z):
    expZ = np.exp(Z - np.max(Z, axis=1, keepdims=True))
    return expZ / expZ.sum(axis=1, keepdims=True)

# 3. INITIALIZE 2 LAYERS OF WEIGHTS (Input -> Hidden -> Output)
def initialize_deep_parameters():

    # Layer 1: 784 inputs connecting to 128 hidden neurons
    W1 = np.random.randn(784, 128) * np.sqrt(2. / 784) # Smarter initialization (He Init)
    b1 = np.zeros((1, 128))

    # Layer 2: 128 hidden neurons connecting to 10 final outputs
    W2 = np.random.randn(128, 10) * np.sqrt(2. / 128)
    b2 = np.zeros((1, 10))

    return W1, b1, W2, b2

# 4. FORWARD PASS (Making the guess)
def forward_deep(X, W1, b1, W2, b2):
    # Pass through Layer 1 and apply ReLU
    Z1 = np.dot(X, W1) + b1
    A1 = relu(Z1)

    # Pass through Layer 2 and apply Softmax
    Z2 = np.dot(A1, W2) + b2
    A2 = softmax(Z2)

    return Z1, A1, Z2, A2

# 5. BACKWARD PASS (The Chain Rule to update both layers)
def update_deep_parameters(X, Z1, A1, A2, Y, W1, b1, W2, b2, learn_rate):
    m = X.shape[0]

    # Layer 2 adjustments (Same as our old model)
    dZ2 = A2 - Y
    dW2 = (1 / m) * np.dot(A1.T, dZ2)
    db2 = (1 / m) * np.sum(dZ2, axis=0, keepdims=True)

    # Layer 1 adjustments (Calculus chain rule through the ReLU function)
    dA1 = np.dot(dZ2, W2.T)
    dZ1 = dA1 * relu_derivative(Z1)
    dW1 = (1 / m) * np.dot(X.T, dZ1)
    db1 = (1 / m) * np.sum(dZ1, axis=0, keepdims=True)

    # Update weights
    W1 -= learn_rate * dW1
    b1 -= learn_rate * db1
    W2 -= learn_rate * dW2
    b2 -= learn_rate * db2

    return W1, b1, W2, b2

In [4]:
# 6. MINI-BATCH TRAINING LOOP
W1, b1, W2, b2 = initialize_deep_parameters()
epochs = 30
batch_size = 64
learn_rate = 0.1

print("Starting Deep Network Training...")
for epo in range(epochs):
    # Shuffle data
    permutation = np.random.permutation(x_train.shape[0])
    x_train_shuf = x_train[permutation]
    y_train_shuf = y_train_encod[permutation]

    for i in range(0, x_train.shape[0], batch_size):
        x_batch = x_train_shuf[i : i + batch_size]
        y_batch = y_train_shuf[i : i + batch_size]

        # Forward and Backward passes
        Z1, A1, Z2, A2 = forward_deep(x_batch, W1, b1, W2, b2)
        W1, b1, W2, b2 = update_deep_parameters(x_batch, Z1, A1, A2, y_batch, W1, b1, W2, b2, learn_rate)

    # Check Accuracy at end of epoch
    _, _, _, final_pred = forward_deep(x_test, W1, b1, W2, b2)
    accuracy = np.mean(np.argmax(final_pred, axis=1) == y_test) * 100
    print(f"Epoch {epo+1}/{epochs} - Test Accuracy: {accuracy:.2f}%")

Starting Deep Network Training...
Epoch 1/30 - Test Accuracy: 93.30%
Epoch 2/30 - Test Accuracy: 95.20%
Epoch 3/30 - Test Accuracy: 96.01%
Epoch 4/30 - Test Accuracy: 96.38%
Epoch 5/30 - Test Accuracy: 96.99%
Epoch 6/30 - Test Accuracy: 96.91%
Epoch 7/30 - Test Accuracy: 96.72%
Epoch 8/30 - Test Accuracy: 97.35%
Epoch 9/30 - Test Accuracy: 97.52%
Epoch 10/30 - Test Accuracy: 97.56%
Epoch 11/30 - Test Accuracy: 97.51%
Epoch 12/30 - Test Accuracy: 97.64%
Epoch 13/30 - Test Accuracy: 97.48%
Epoch 14/30 - Test Accuracy: 97.79%
Epoch 15/30 - Test Accuracy: 97.63%
Epoch 16/30 - Test Accuracy: 97.67%
Epoch 17/30 - Test Accuracy: 97.75%
Epoch 18/30 - Test Accuracy: 97.66%
Epoch 19/30 - Test Accuracy: 97.77%
Epoch 20/30 - Test Accuracy: 97.63%
Epoch 21/30 - Test Accuracy: 97.86%
Epoch 22/30 - Test Accuracy: 97.81%
Epoch 23/30 - Test Accuracy: 97.85%
Epoch 24/30 - Test Accuracy: 97.92%
Epoch 25/30 - Test Accuracy: 97.80%
Epoch 26/30 - Test Accuracy: 97.98%
Epoch 27/30 - Test Accuracy: 97.95%
Epo